In [1]:
"""
Project Title: Elevation-based NDVI Analysis
Author: James McLoughlin
Date: May 2026
Description: This script processes DEM and Landsat data to calculate zonal statistics for vegetation health across different elevation zones.
"""

# =============================================================================
# Library Imports
# =============================================================================
# Standard library imports
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # silences future development warninigs

# # Geospatial and Data Science libraries
import earthaccess           # For NASA Earthdata access and authentication
import geopandas as gpd      # For vector data manipulation
import numpy as np           # For numerical array operations
import pandas as pd          # For tabular data management and CSV export
from pathlib import Path     # For file path handling
import rasterio as rio       # Core library for reading and writing raster datasets
import rasterio.merge
from rasterio import features
from rasterio.merge import merge
from rasterstats import zonal_stats # For calculating statistics within polygons
import rioxarray             # Extension for xarray to handle geospatial coordinates
import shapely               # For manipulation and analysis of geometric objects
from shapely.geometry import shape
import xarray as xr          # For handling multi-dimensional arrays


# =============================================================================
# User Inputs and configuration
# =============================================================================
# Define the root directory for all project data to maintain relative paths
base = Path("C:/Users/jj_mc/OneDrive/Documents/GitHub/EGM722-Assessment/Data")
step = 500

# =============================================================================
# Helper Functions
# =============================================================================
def raster_dataframe():
    """
    Searches a presepcified directory and sub directories to catalog Landsat 8 and 9 satellite imagery.
    
    This function iterates through specified data path, extracts image specific infotmation from 
    the Landsat filenames and maps bands the band number to the relative spectral color. It compiles 
    the information into a Pandas dataframe.

    Returns:
        pd.DataFrame: A DataFrame containing columns for filename, satellite ID, 
                      acquisition year, band ID, spectral colour, and file path.
    """
    records = []
    
    # a. Mapping Landsat 8/9 band identifiers to spectral colours 
    l8_9_bands = {
        "B1":"coast/aero", "B2":"blue", "B3":"green", "B4":"red", "B5":"nir", "B6":"swir1", "B7":"swir2"
    }
    
    # b. Iterates through folders within the configured Landsat data directory
    for folder in PATHS["landsat_data"].iterdir():
        if folder.is_dir():
            # Target Surface Reflectance (SR) TIF files
            for file in folder.glob("*_SR_B*.TIF"):
                parts = file.name.split("_")
                sat_id = parts[0][:4]  # Extract satellite ID, e.g. 'LC08'
                band_id = parts[-1].replace(".TIF", "")  # Extract band number, e.g. 'B1'

                # Assigns spectral color based on the satellite band mapping
                if sat_id in ["LC08", "LC09"]:
                    colour = l8_9_bands.get(band_id)

                # Only includes the files if the colour is required forthe study
                if colour in ALL_COLOURS:
                    records.append({
                        "filename": file.name,
                        "satellite": sat_id,
                        "year": parts[3][:4],
                        "band": band_id,
                        "colour": colour,
                        "path": str(file)
                    })
    
    # c. Return the collected metadata as a pandas DataFrame for analysis
    return pd.DataFrame(records)
    

def create_mosaic(file_list, out_path, dtype=None, nodata_val=None):
    """
    Merges multiple files into a single mosaicked image.

    Args:
        file_list (list): List of file paths (Path objects or strings) of the images to be merged.
        out_path (Path): Destination path for the generated mosaic (in .TIF formate).
        dtype (str, optional): Desired data type for output. Defaults to source dtype.
        nodata_val (float, optional): Value representing 'no data'. Defaults to source value.

    Returns:
        Path: The file path where the mosaic was successfully saved.
    """
    # a. Access the first file to establish baseline spatial metadata (CRS and Profile)    
    with rio.open(file_list[0]) as src:
        target_crs = src.crs
        out_meta = src.profile.copy()
        if nodata_val is None:
            nodata_val = src.nodata
        
    # b. Create the list of files to be merge.
    to_merge = []
    for f in file_list:
        with rio.open(f) as src:
                to_merge.append(f)
               
    # c. Mosiack the files, returns the combined array and new geotransform
    mosaic, out_trans = merge(to_merge, nodata=0)

    # d. File clean-up
    for s in to_merge:
        if hasattr(s, 'close'):
            s.close()
   
    # e. Synchronize metadata with the new mosaic dimensions and transformation
    bands, height, width = mosaic.shape
    out_meta.update({
        "height": height, 
        "width": width, 
        "transform": out_trans, 
        "nodata": 0,
        "dtype": dtype or out_meta['dtype']
    })

    # f. Write the array to disk
    with rio.open(out_path, "w", **out_meta) as dest:
        dest.write(mosaic)
    print(f"     Mosaic saved: {out_path.name}")
    return out_path   # path used to build the dataset used for analysis 


# =============================================================================
# Main Worflow
# =============================================================================

# --- 1. Create folder structure ---
# =============================================================================
# Define project directory strcutre using pathlib
PATHS = {
    "landsat_data" : base / "Landsat_Rasters",
    "mosaics": base / "Mosaics",
    "vectors": base / "Vector_Layers",
    "earthaccess": base / "EarthAccess",
    "results": base / "Results"
}

# Batch directory creation, and checks parent exisits to ensure full path creation 
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)
print("1. Directories checked/created")



# --- 2. Colours bands required --- 
# =============================================================================
# The composite images and analysis tasks to be conducteed
DISPLAY_TASKS = ("true_colour","false_colour")
ANALYSIS_TASKS = ("ndvi",)

TASK_BANDS = {
    "true_colour" : ["red", "green","blue"],
    "false_colour" : ["nir","red","green"],
    "ndvi": ["red", "nir"],
    "ndwi": ["nir","green"],
    "ndsi": ["swir1","green"]
}

# Identify unique bands required across for analysis / display tasks, sets automatically handle de-duplication.
ANALYSIS_COLOURS = {band for task in ANALYSIS_TASKS for band in TASK_BANDS[task]}
DISPLAY_COLOURS = {band for task in DISPLAY_TASKS for band in TASK_BANDS[task]}

# Combine analysis and display colours into a master of unique colours using set union (|)
ALL_COLOURS = list(DISPLAY_COLOURS | ANALYSIS_COLOURS)
print("2. Band identification and mapping completed")



# --- 3. Build DataFrame of rasters needed for mosaicking ---
# =============================================================================
# Generate a Pandas dataframe of available local imagery matching the required bands
raster_df = raster_dataframe()

# Extract the year of the data being analysed
current_year = raster_df['year'].unique()[0]

# Export the dataframe to CSV
raster_df.to_csv(PATHS["landsat_data"] / f"{current_year}_raster_dataframe.csv", index=False)
print(F"3. Image dataset created : {len(raster_df)} files included")



# --- 4. Mosaick Rasters --- 
file_map = {}
print("4. Mosaicking started...")

# Group the dataframe by color to process all relevant tiles together
for colour, group in raster_df.groupby('colour'):
    # Extract file paths into a list for mosaicking
    file_list = group['path'].tolist()     
    
    # Define the output destination of the mosiac
    dst_path = PATHS["mosaics"] / f"{current_year}_{colour}_mosaic.tif"
    
    # Call mosaic function to merge tiles listed in file_lits
    # Returns the path of the mosaicked image, to populate file_map for use in Section 6
    mosaic_path = create_mosaic(file_list, dst_path, nodata_val=0)     
    
    # Store file path in a dictionary for easy access during Dataset assembly
    file_map[colour.lower()] = mosaic_path 

print("     All rasters mosaicked")



# --- 5. Build Dataset of Mosaicked Images ---
# =============================================================================
mosaic_list = []

# Iterate through the file-map established in the previous section
for name, path in file_map.items():
    
    # Load rasters using rioxarray with Dask-backed chunking.
    # Uses 'Lazy Loading' to process pieces of the raster as needed and prevent memory saturation.
    mosaic_da = rioxarray.open_rasterio(path, chunks={'x': 2048, 'y': 2048})
    
    # Pre-process DataArray for Dataset integration:
    # 1. .squeeze() removes the single 'band' dimension, resulting in a 2D array.
    # 2. .drop_vars() removes non-spatial coordinates that may conflict during merging.
    # 3. .rename() assigns the spectral band name (e.g., 'red') to the variable.
    mosaic_da = mosaic_da.squeeze().drop_vars('band', errors='ignore').rename(name)
    mosaic_list.append(mosaic_da)

# Combine individual arrays into a single multi-dimensional xarray Dataset
ds = xr.merge(mosaic_list)

print(f"5. Dataset for {current_year} completed")



# --- 6. NDVI Analysis ---
# =============================================================================
# Tmo minmise memory usage reduce the rasters to the region of interst
# Locate and read the Region of Interest (ROI) vector file
roi = PATHS["vectors"] / "national_park_boundary.shp"
roi_gdf = gpd.read_file(roi)

# Reproject the ROI to match the dataset CRS
roi_gdf = roi_gdf.to_crs(ds.rio.crs)

# Clip the dataset to the ROI boundary and remove empty 'bounding box' pixels outside the polygon (drop=True)
ds_clipped = ds.rio.clip(roi_gdf.geometry, ds.rio.crs, drop=True) 

# Mask out zero values to avoid divisde by zero errors during calculation
ds_clean = ds_clipped.where(ds_clipped > 0)

# Iterate through active analysis tasks (e.g., NDVI, NDWI)
for task in ANALYSIS_TASKS: 
    # Retrieve required bands for the current index calculation
    b1_name = TASK_BANDS[task][0] 
    b2_name = TASK_BANDS[task][1] 

    # Concert to float to avoid calaucation errors (Landsat 8/9 repreaented as uint16)
    b1 = ds_clean[b1_name].astype(float)
    b2 = ds_clean[b2_name].astype(float)

    # Calaualted the Normalised Differnce Index
    ds_clipped[task] = (b2-b1)/(b2+b1)

    # Remove and points that have inf value (i.e. were /0)
    ds_clipped[task] = ds_clipped[task].where(np.isfinite(ds_clipped[task])) 
print("6. Normalized Difference Index Analysis Completed")



# --- 7. Acquire DEMs from Earth Access---
# =============================================================================
# DEMs required for elevation based analysis. ROI polygon used to spatially slect DEMs from NASA EarthAceess
# Reproject the ROI polygon to CRS requied for EarthAccess (EPSG:4326)
roi_4326 = roi_gdf.to_crs(epsg = 4326).union_all()

# Create mimnimum rotated retangle around ROI polygon (siple polygon needed for EarthAcress data search)
search_area = roi_4326.minimum_rotated_rectangle

# Change polygon vertices definiton to counter-clockwise orientation required by EarthAcess
search_area = shapely.geometry.polygon.orient(search_area, sign=1) 

print("7. EarthAccess data acquistion started...")

# Log into EarthAccess using credentials in local .netric file
earthaccess.login(strategy='netrc') 

# Searh the ASTER Global Digital Elevation Model (ASTGTM) collection
# Filter results using exterior coordinates of ROI's mimnimum rotated retangle
earthaccess_files = earthaccess.search_data( 
    short_name = 'ASTGTM',
    polygon = search_area.exterior.coords
)

# Execute batch download to the local pre-defined directory
earthaccess_dest = PATHS["earthaccess"]
downloaded_files = earthaccess.download(earthaccess_files, earthaccess_dest)



# --- 8. Mosaicking the DEMs ---
# =============================================================================
print("8. Mosaicking DEMs...")

# Filter the downloaded files to target only the Digital Elevation Model TIFs
dem_files = [fn for fn in downloaded_files if 'dem.tif' in fn.name]

# Define destination for the mosaikced DEM 
dem_mosaic_path = PATHS["mosaics"] / "DEM_mosaic.TIF" 

# Call 'create_mosaic' helper fuction to merge and save mosaic. 
create_mosaic(dem_files, dem_mosaic_path, dtype='float32')  #dtype set to 'float32' to minimise memory usage 



# --- 9. Vectorising Mosaicked DEM ---
# =============================================================================
# Extracting polygons from the DEM describing zones with different elevations
print("9. Vectorising DEM...")

# Load the DEM mosaic and remove the single band dimension
dem_master = rioxarray.open_rasterio(dem_mosaic_path).squeeze() 

# Align the ROI coordinate system to match the DEM for accurate clipping
roi_dem = roi_gdf.to_crs(dem_master.rio.crs)

# Clip the DEM to the ROI to minimise memory usage during vectorization
dem_clipped = dem_master.rio.clip(roi_dem.geometry, drop=True)

# Segment the elevation data into discrete elevation zones using floor division
# dtype "int32" used to minimise memory usage 
dem_segmented = (dem_clipped.values // step).astype("int32")

# Create a boolean mask to exclude NaN/NoData data from the vectorization
mask = ~np.isnan(dem_segmented)

# Extract vector shapes from the segmented raster data
shapes = features.shapes(dem_segmented, mask=mask,transform=dem_clipped.rio.transform())

polygons = [] 
for i, (geom, value) in enumerate(shapes): 
    # Scale segmented value back to true elevation and store with unique ID and geometry
    polygons.append({
        'poly_id': i,
        'elevation_zone': int(value * step), 
        'geometry': shape(geom)
    })

# Convert list of dictionaries into a GeoDataFrame using the clipped DEM's CRS
segments_gdf = gpd.GeoDataFrame(polygons, crs=dem_clipped.rio.crs)

# Reproject the polygons to match the Landsat CRS for consistent spatial analysis
segments_gdf = segments_gdf.to_crs(ds_clipped.rio.crs)

# Define output path and save the vector polygons as a .gpkg
output_path = PATHS["vectors"] / "elevation_zones.gpkg"
segments_gdf.to_file(output_path, driver="GPKG")

print("      Vectorising completed and saved")



# --- 10. Spatial Analysis & Data Integration ---
# =============================================================================
# Calculate zonal statistics for vegetation health (NDVI) across elevation zones
print("10. Calculating Zonal Statistics...")

# Extract the NDVI layer from dataset for the current year
ndvi_layer = ds_clipped.ndvi

# Execute zonal statistics to compute NDVI mean and standard deviation per polygon
stats = zonal_stats(
    segments_gdf,
    ndvi_layer.values,
    affine=ds_clipped.rio.transform(),
    stats=['mean', 'std'],
    nodata=np.nan,
    all_touched=True
)

# Convert statistics into a DataFrame and clean numeric types
stats_df = pd.DataFrame(stats)
stats_df['poly_id'] = segments_gdf['poly_id'] # Maintain link to polygons

# Join the statistics directly onto the GeoDataFrame for integrated storage
segments_gdf = segments_gdf.merge(stats_df, on='poly_id')

# Add metadata for temporal tracking
segments_gdf['year'] = current_year

# Save the integrated vector dataset (polygons + statistics) as a GeoPackage
output_file = PATHS["results"] / f"elevation_zones_with_stats_{current_year}.gpkg"
segments_gdf.to_file(output_file, driver="GPKG")

# Export a standalone CSV for reporting and non-spatial analysis
output_csv = PATHS["results"] / f"final_stats_{current_year}.csv"
segments_gdf.drop(columns='geometry').to_csv(output_csv, index=False)

print(f"      Zonal statistics integrated and saved")


# --- 11. Create Composite Images ---
# =============================================================================
print("11. Creating composite images")

# Iterate through each active display task (e.g., 'true_colour', 'false_colour')
for disp_tsk in DISPLAY_TASKS:
    
    # Slice the multi-variable Dataset to specific bands required for the task
    # .to_array() converts the specific bands raters into a multi-band composite
    img = ds[TASK_BANDS[disp_tsk]].to_array(dim='band')
    
    # Define destination for the composite array.
    img_dest = PATHS["results"] / f"{current_year}_{disp_tsk}_composite.tif"
    
    # Saves the array in a .tif format using the rioxarray extension
    img.rio.to_raster(img_dest)
    
    print(f"     Composite saved: {current_year} {disp_tsk}")

    
print("--- Code completed ---")

1. Directories checked/created
2. Band identification and mapping completed
3. Image dataset created : 8 files included
4. Mosaicking started...
     Mosaic saved: 2025_blue_mosaic.tif
     Mosaic saved: 2025_green_mosaic.tif
     Mosaic saved: 2025_nir_mosaic.tif
     Mosaic saved: 2025_red_mosaic.tif
     All rasters mosaicked
5. Dataset for 2025 completed
6. Normalized Difference Index Analysis Completed
7. EarthAccess data acquistion started...


QUEUEING TASKS | :   0%|          | 0/16 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/16 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/16 [00:00<?, ?it/s]

8. Mosaicking DEMs...
     Mosaic saved: DEM_mosaic.TIF
9. Vectorising DEM...
      Vectorising completed and saved
10. Calculating Zonal Statistics...
      Zonal statistics integrated and saved
11. Creating composite images
     Composite saved: 2025 true_colour
     Composite saved: 2025 false_colour
--- Code completed ---
